# Análisis del baseline SINCA — notebook base

Notebook de arranque para **análisis puntuales** sobre los resultados del baseline (Fase 3). Corre dentro del Space `sinca-dev` de SageMaker Studio (dominio `sinca-studio`), que ejecuta como rol `sinca-dev-role`: **no hace falta (ni se debe) escribir credenciales AWS**.

> **Encender**: SageMaker Console → Studio → dominio `sinca-studio` → Spaces → `sinca-dev` → **Open Studio** (JupyterLab).
> **Trabajar**: todo este notebook.
> **Apagar**: SageMaker Console → Studio → Spaces → `sinca-dev` → **Stop**. Solo con la app detenida dejás de pagar cómputo (ver sección final).

## Fuentes de datos disponibles

- `s3://sinca-data/validated/` — series validadas (SINCA, Open-Meteo, feriados).
- `s3://sinca-mlflow/_mlflow/mlflow.sqlite` — métricas/parámetros por run.
- `s3://sinca-mlflow/_mlflow/mlruns/.../artifacts/predictions.parquet` — predicciones por serie (tras el re-run del baseline).

In [ ]:
import io
import os
import sqlite3
import tempfile

import boto3
import pandas as pd
import matplotlib.pyplot as plt

BUCKET_DATA = "sinca-data"
BUCKET_MLFLOW = "sinca-mlflow"
MLFLOW_PREFIX = "_mlflow"

# boto3 usa automáticamente el rol del Space (sinca-dev-role). No usar profile ni claves.
s3 = boto3.client("s3")

In [ ]:
# Descarga el backend SQLite de MLflow y lista los runs.
tmp = tempfile.mkdtemp()
db = os.path.join(tmp, "mlflow.sqlite")
s3.download_file(BUCKET_MLFLOW, f"{MLFLOW_PREFIX}/mlflow.sqlite", db)

con = sqlite3.connect(db)
runs = pd.read_sql_query("SELECT run_name, status, start_time FROM runs ORDER BY start_time DESC", con)
metrics = pd.read_sql_query("SELECT run_uuid, key, value FROM metrics", con)
con.close()

display(runs.head(20))
display(metrics.head(20))

In [ ]:
# Lista e inspecciona los Parquet validados de SINCA (ejemplo).
resp = s3.list_objects_v2(Bucket=BUCKET_DATA, Prefix="validated/sinca/", MaxKeys=20)
keys = [o["Key"] for o in resp.get("Contents", []) if o["Key"].endswith(".parquet")]
print(keys[:10])

if keys:
    body = s3.get_object(Bucket=BUCKET_DATA, Key=keys[0])["Body"].read()
    df = pd.read_parquet(io.BytesIO(body))
    display(df.head())

In [ ]:
# Localiza y lee las predicciones persistidas de los runs (columna predictions.parquet).
listing = s3.list_objects_v2(Bucket=BUCKET_MLFLOW, Prefix=f"{MLFLOW_PREFIX}/mlruns/")
all_keys = [o["Key"] for o in listing.get("Contents", [])]
pred_keys = [k for k in all_keys if k.endswith("artifacts/predictions.parquet")]
print(pred_keys)

pred = None
if pred_keys:
    body = s3.get_object(Bucket=BUCKET_MLFLOW, Key=pred_keys[0])["Body"].read()
    pred = pd.read_parquet(io.BytesIO(body))
    display(pred.head())

In [ ]:
# Plot rápido de ejemplo: serie real vs pred (si hay predicciones).
if pred is not None and not pred.empty:
    one = pred.iloc[0]
    serie = pred[(pred.estacion == one.estacion) & (pred.pollutant == one.pollutant)].sort_values("timestamp")
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(serie.timestamp, serie.y_true, label="real", lw=1)
    ax.plot(serie.timestamp, serie.y_pred, label="pred", lw=1, alpha=0.8)
    ax.set_title(f"{one.estacion} / {one.pollutant} — real vs pred")
    ax.legend()
    plt.show()

## Al terminar

1. Guardá este notebook (Ctrl+S). Queda en el **EBS de 5 GB** del Space: persiste aunque apagues.
2. **Apagá el Space**: SageMaker Console → Studio → Spaces → `sinca-dev` → **Stop**.

> Con la app detenida, `aws sagemaker list-apps` queda vacío y **deja de facturar cómputo** (~USD 0.05–0.08/h del `ml.t3.large` mientras estuviera encendida). El EBS sigue cobrando centavos (~USD 0.40/mes).